In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import pandas as pd
from pathlib import Path

from PIL import Image
import torch

import warnings
warnings.filterwarnings("ignore")

In [ ]:
DATA_DIR = Path.cwd().parent.parent.parent.resolve() / "data"
metadata_dir = DATA_DIR / "GF1_B02_B03_B04_B08_Mask"
metadata_path = metadata_dir / "metadata.csv"

assert metadata_path.exists()

In [ ]:
df = pd.read_csv(metadata_path)
df.head()

In [ ]:
metadata = list(metadata_dir.glob("*.tif"))
metadata

In [ ]:
def update_path(row):
    file_suffix = ".tif"
    filename = row["filename"] + file_suffix
    path = os.path.join(metadata_dir, filename)
    return path

df["path"] = df.apply(update_path, axis=1)
df = df[df["path"].apply(os.path.exists)]
df.head(100)

In [ ]:
CSV_OUT_DIR = DATA_DIR / "metadata"
IMG_OUT_DIR = DATA_DIR / "images"
LABEL_OUT_DIR = DATA_DIR / "labels"
PRED_OUT_DIR = DATA_DIR / "predictions"

In [ ]:
import sys
from pathlib import Path

notebook_dir = Path.cwd()
root_dir = notebook_dir.parent.parent
sys.path.append(str(root_dir))

root_dir

# Generate training chips

In [ ]:
training_targets_txt = CSV_OUT_DIR / "training_targets.txt"
with open(training_targets_txt, 'r', encoding='utf-8') as f:
    training_targets = [line.strip() for line in f if line.strip()]
training_targets

In [ ]:
df_for_training = df[df["filename"].isin(training_targets)].copy()
df_for_training

In [ ]:
from benchmark.core.geotiff_tiler import GeoTIFFTiler
from benchmark.inference import predict_full_geotiff
import traceback

In [ ]:
try:
    for i in range(len(df_for_training)):
        t = GeoTIFFTiler(df_row=df_for_training.iloc[i], display_thumbnail=True)
        t.get_chips(CSV_OUT_DIR, IMG_OUT_DIR, LABEL_OUT_DIR)
except Exception as e:
    traceback.print_exc()

# Generate testing chips

In [ ]:
testing_targets_txt = CSV_OUT_DIR / "testing_targets.txt"
with open(testing_targets_txt, 'r', encoding='utf-8') as f:
    testing_targets = [line.strip() for line in f if line.strip()]
testing_targets

In [ ]:
df_for_testing = df[df["filename"].isin(testing_targets)].copy()
df_for_testing

In [ ]:
from benchmark.configs import Configs

try:
    config = Configs.balanced()
    config.batch_size = 16
    config.use_tta = True
    model_weights_path = Path(config.model_name) / Path("assets/cloud_model.pt")
    for i in range(len(df_for_testing)):
        t = GeoTIFFTiler(df_row=df_for_testing.iloc[i], display_thumbnail=True)
        t.get_chips(CSV_OUT_DIR, IMG_OUT_DIR, LABEL_OUT_DIR)
        predict_full_geotiff(t, PRED_OUT_DIR, model_weights_path=model_weights_path, config=config)
except Exception as e:
    traceback.print_exc()

In [ ]:
data_dir = DATA_DIR / "GF1_B02_B03_B04_B08_Mask"
data_tif_files = list(data_dir.glob("*.tif"))
data_tif_files

In [ ]:
pred_dir = DATA_DIR / "predictions"
pred_tif_files = list(pred_dir.glob("*_PredictedMask.tif"))
pred_tif_files

In [ ]:
from benchmark.visualization import display_thumbnail_with_prediction

try:
    for data_tif_file in data_tif_files:
        for pred_tif_file in pred_tif_files:
            filename = str(data_tif_file.stem)
            if filename in str(pred_tif_file) and filename in testing_targets:
                display_thumbnail_with_prediction(data_tif_file, pred_tif_file)
except Exception as e:
    traceback.print_exc()